# Notebook d'introduction à Scikit-learn

1. Présentation très rapide des différents cadres de machine-learning
2. Transposition dans l'univers `sklearn` (reposant lui-même sur `numpy` / `matplotlib`)
3. Prise en main des outils

**Progression du notebook**: on part des *données* (A), on met en oeuvre des *modèles simples* que l'on sait
visualiser et comprendre en 2D (B), on passe aux *modèles de l'état de l'art* (C), on montre comment *étendre*
la bibliothèque avec ses propres modèles (D), puis on consolide avec des *exercices complémentaires* (E).

**Conventions**: les cellules à compléter sont signalées par un commentaire `TODO` dans la version étudiante ;
les blocs `<span style="color:red">Mini-exo</span>` sont de courts exercices d'application immédiate.

<a id="sec-plan"></a>
## Plan

* [A. Les données](#sec-a)
    * [A.1 Un jeu de données réel: iris](#sec-a1)
    * [A.2 Des données jouets en 2D](#sec-a2)
    * [A.3 Séparation apprentissage / test](#sec-a3)
* [B. Classification supervisée](#sec-b)
    * [B.1 Mise en oeuvre des modèles](#sec-b1)
    * [B.2 Evaluation & comparaison de modèles](#sec-b2)
    * [B.3 Analyse qualitative sur les données jouets](#sec-b3)
    * [B.4 Introspection des modèles](#sec-b4)
    * [B.5 Spécificité des modèles](#sec-b5)
    * [B.6 Jouons avec un arbre de décision](#sec-b6)
    * [B.7 Jouons avec le SVM](#sec-b7)
* [C. Modèles de l'état de l'art](#sec-c)
    * [C.1 Premières expérimentations](#sec-c1)
    * [C.2 Introspection: pondération des caractéristiques](#sec-c2)
* [D. Extension de scikit-learn](#sec-d)
* [E. Exercices complémentaires](#sec-e)
    * [E.1 Les k plus proches voisins](#sec-e1)
    * [E.2 Mise à l'échelle des données et pipelines](#sec-e2)
    * [E.3 Courbe de complexité: sous-apprentissage et sur-apprentissage](#sec-e3)
    * [E.4 Validation croisée et recherche d'hyper-paramètres](#sec-e4)
    * [E.5 La même API pour la régression](#sec-e5)
    * [E.6 Synthèse: comparaison de modèles sur USPS](#sec-e6)
* [Annexe: transformation du notebook en version étudiante](#sec-annexe)

In [ ]:
# Les incontournables
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl

# scikit-learn est découpé en (nombreux) sous-modules: on importe au fur et à mesure des besoins
from sklearn import datasets

# reproductibilité: on fixe la graine du générateur pseudo-aléatoire
np.random.seed(42)

<a id="sec-a"></a>
## A. Les données

Quelques jeux de données classiques sont déjà dans la boite à outils, on va les utiliser pour aller plus vite.
Évidemment, l'enjeu est ensuite de passer à des jeux de données plus proches de la réalité (e.g. challenge Kaggle)
ou à n'importe quel problème réel en important des données (csv, json, xls, ...) à l'aide des fonctions existantes.

<a id="sec-a1"></a>
### A.1 Un jeu de données réel: iris

Le jeu `iris` est le *hello world* de la classification: 150 fleurs, 4 mesures (caractéristiques) par fleur,
3 espèces à reconnaitre (étiquettes).

In [ ]:
# spécifique à scikit-learn: les jeux de données "clés en main"
iris = datasets.load_iris()

# les données sont dans un dictionnaire python: on regarde d'abord les clés
print("Structuration des données  : ", list(iris.keys()))
# puis les dimensions des valeurs d'intérêt
print("Dimension de X (n, d)      : ", iris.data.shape)    # n individus x d caractéristiques
print("Dimension de Y (n,)        : ", iris.target.shape)  # 1 étiquette par individu
print("Noms des caractéristiques  : ", iris.feature_names)
print("Noms des classes           : ", iris.target_names)

In [ ]:
# histoire de rendre les choses plus concrètes:
# tracé des deux premières caractéristiques (la couleur code l'étiquette)

X = iris.data
Y = iris.target

plt.figure(facecolor='white')
plt.scatter(X[:,0], X[:,1], c=Y)   # c accepte un vecteur de nombres = un code couleur
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title("Iris: 2 caractéristiques sur les 4 disponibles")
plt.grid()

<a id="sec-a2"></a>
### A.2 Des données jouets en 2D

Afin de mieux comprendre la suite, on propose de travailler sur des données jouets, en 2D: ces données sont
directement visualisables et permettent de comprendre le fonctionnement interne des classifieurs.
Ces données `X, Y` nous serviront pendant toute la partie B.

**Attention** à toujours garder en tête le coté *factice* des données 2D et à faire l'effort mental de
transposer vos conclusions en plus haute dimension.

In [ ]:
n   = 100    # nombre de points
sig = 1.5    # écart-type de chaque nuage

np.random.seed(42)   # (re)fixée ici pour que la cellule soit rejouable seule
X = np.random.randn(n,2)*sig
Y = np.zeros(n)
Y[n//2:] = 1         # une moitié des données en y=0, l'autre en y=1
X[Y==0] += 2         # décalage des deux classes l'une par rapport à l'autre
X[Y==1] -= 2

plt.figure(facecolor='white')
plt.scatter(X[:,0], X[:,1], c=Y)
plt.title("Données jouets: 2 gaussiennes en 2D")
plt.grid()

<a id="sec-a3"></a>
### A.3 Séparation apprentissage / test

Un modèle qui n'est évalué que sur les données qui ont servi à l'apprendre ne prouve rien: il faut **toujours**
mesurer les performances sur des données jamais vues pendant l'apprentissage.

Nous faisons ici le découpage *à la main* (avec `numpy`) pour bien comprendre ce qui se passe ;
la fonction `train_test_split` de `sklearn` fera le travail à notre place à partir de la partie B.7.

In [ ]:
# séparation des indices d'apprentissage et de test
np.random.seed(1)                        # rejouabilité de la cellule
pcapp   = 0.7                            # 70% des données pour l'apprentissage
ind     = np.random.permutation(len(X))  # tirage aléatoire SANS remise des indices
indapp  = ind[:int(pcapp*len(X))]
indtest = ind[int(pcapp*len(X)):]

Xapp,  Yapp  = X[indapp],  Y[indapp]
Xtest, Ytest = X[indtest], Y[indtest]

print("Apprentissage:", Xapp.shape, "  Test:", Xtest.shape)

In [ ]:
# [Mini-exo] Visualiser les données d'apprentissage ET de test sur la même figure,
#            avec un code de forme différent:
#            couleur = classe (jaune / violet), rond = apprentissage, étoile = test

plt.figure(facecolor='white')
#  TODO 
plt.title("Découpage apprentissage / test")
plt.grid()

<a id="sec-b"></a>
## B. Classification supervisée

Commençons par étudier quelques modèles classiques en classification supervisée...

<a id="sec-b1"></a>
### B.1 Mise en oeuvre des modèles

Il s'agit de programmation objet et d'héritage... Mais pour l'utilisateur, c'est surtout un ensemble de modèles
facilement disponibles sur l'étagère, **tous pilotés de la même manière**:

1. Initialisation du modèle & de ses hyper-paramètres (= création du classifieur)
2. Apprentissage sur les données d'entrainement (= `fit`)
3. Inférence sur les données d'apprentissage ou de test (= `predict`)<BR>
  ATTENTION: `predict` attend un *ensemble* de données, pas un individu isolé

In [ ]:
from sklearn import svm, linear_model, naive_bayes

# 1. création d'un modèle bayésien naïf (aucune donnée n'est vue à ce stade)
mod = naive_bayes.GaussianNB()
# 2. apprentissage sur les données d'apprentissage
mod.fit(Xapp, Yapp)
# 3. inférence sur un individu (le premier point de test)
yhat = mod.predict([Xtest[0]])      # ATTENTION: la fonction est prévue pour traiter un
                                    # ENSEMBLE de points et pas une donnée isolée
                                    # d'où les [] supplémentaires
                                    # => elle retourne donc une liste de prédictions

print("comparaison entre prédiction et vérité terrain: ", yhat, Ytest[0])

In [ ]:
# Parfois on veut prédire une CLASSE, parfois on veut un SCORE
# 1. classe => plus simple pour les métriques (e.g. taux de bonne classification)
# 2. score  => indispensable pour mesurer une confiance, une distance à la frontière, ...

yhat  = mod.predict([Xtest[0]])
score = mod.predict_proba([Xtest[0]])   # predict_proba existe aussi sur les modèles non bayésiens
                                        # ATTENTION: un score PAR CLASSE (dim != yhat)

print("classe prédite    :", yhat)
print("scores par classe :", score)

#### <span style="color:red">Mini-exo</span>: rappels de numpy

1. Calculer *à la main* le taux de bonne classification à partir d'une sortie probabiliste
2. En admettant un taux de rejet des points les plus ambigus (5%), que devient le taux de bonne classification
   sur les 95% restants?

**Rappel**: il s'agit d'une technique de rejet qui peut être très utile en contexte opérationnel
(on préfère ne pas répondre plutôt que de répondre n'importe quoi).

In [ ]:
yhat_prob = mod.predict_proba(Xtest)     # (n_test, n_classes)

# 1. calculer le taux de bonne classification par rapport à Ytest
#  TODO 

# 2. trouver les indices des 95% des points les moins ambigus,
#    puis calculer le taux de bonne classification sur ces seuls points
#  TODO 

#### <span style="color:red">Mini-exo</span>: passage en plus grande dimension

Le classifieur utilisé `naive_bayes.GaussianNB()` correspond à l'une des solutions envisagées pour les données
USPS (images de chiffres manuscrits, 16x16 pixels) dans un notebook précédent: vérifier que vous êtes capable
d'évaluer ce classifieur sur ces données.

Rien ne change dans le code: seule la dimension des données passe de 2 à 256.

In [ ]:
# 1. chargement des données
data = pkl.load(open("data/usps.pkl",'rb'))
# data est un dictionnaire contenant les champs explicites X_train, X_test, Y_train, Y_test
Xu_train = np.array(data["X_train"], dtype=float)  # changement de type pour éviter les problèmes d'affichage
Xu_test  = np.array(data["X_test"],  dtype=float)
Yu_train = data["Y_train"]
Yu_test  = data["Y_test"]
print("USPS:", Xu_train.shape, Xu_test.shape, " classes:", np.unique(Yu_train))

# 2. apprentissage du modèle
#  TODO 

# 3. Eval: quel taux de bonne classification?
#  TODO 

<a id="sec-b2"></a>
### B.2 Evaluation & comparaison de modèles

Un des enjeux du machine learning consiste à choisir le modèle qui marche le mieux pour un problème donné.
L'architecture de `sklearn` est particulièrement performante pour répondre à cette question: puisque tous les
modèles partagent la même interface, les comparer revient à changer une ligne de code.

In [ ]:
# comparaison simple de modèles
mod1 = naive_bayes.GaussianNB()   # modèle génératif, très rapide
mod2 = svm.SVC()                  # SVM, noyau gaussien par défaut

mod1.fit(Xapp, Yapp)
mod2.fit(Xapp, Yapp)

yhat1 = mod1.predict(Xtest)
yhat2 = mod2.predict(Xtest)

print("perf modèle 1 (NB) ", np.where(yhat1 == Ytest,1,0).mean())
print("perf modèle 2 (SVM)", np.where(yhat2 == Ytest,1,0).mean())

# raccourci: tous les classifieurs possèdent une méthode score (= taux de bonne classification)
print("idem avec .score   ", mod1.score(Xtest, Ytest), mod2.score(Xtest, Ytest))

#### Sélection des hyper-paramètres

La plupart des modèles ont des hyper-paramètres qui impactent beaucoup les performances... cf exemple ci-dessous.
Il faut donc comprendre la sélection de modèle aussi comme une manière d'optimiser ces hyper-paramètres
(cf. exercice [E.4](#sec-e4) pour l'automatisation de cette recherche).

In [ ]:
mod_SVM1 = svm.SVC(kernel="linear", probability=True)  # probability=True pour les affichages 3D ci-dessous
mod_SVM2 = svm.SVC(gamma=10, probability=True)         # noyau gaussien très "serré"

mod_SVM1.fit(Xapp, Yapp)
mod_SVM2.fit(Xapp, Yapp)

yhat1 = mod_SVM1.predict(Xtest)
yhat2 = mod_SVM2.predict(Xtest)

print("perf SVM linéaire        ", np.where(yhat1 == Ytest,1,0).mean())
print("perf SVM gaussien (g=10) ", np.where(yhat2 == Ytest,1,0).mean())

<a id="sec-b3"></a>
### B.3 Analyse qualitative sur les données jouets

Traçons les frontières de décision pour comprendre les modèles... Évidemment, on ne peut tracer ces fonctions
qu'en 2D.

Note 1: la fonction de tracé de la frontière est un peu complexe, pas besoin de comprendre en profondeur
(ou alors demander au prof.)

Note 2: la fonction est donnée... Mais pas dans le notebook: afin de rendre le code plus clair, les fonctions
de tracé sont dans un module externe (= un répertoire avec des fichiers de code).

1. Les modules sont (très) importants en python. Il faut apprendre à les utiliser, puis à en faire:
    * un répertoire contenant un fichier `__init__.py` (vide dans un premier temps)
    * un ou plusieurs fichiers contenant des fonctions
2. L'interface entre un notebook et un module peut être complexe: en effet, par défaut, les modifications du
   module ne sont pas prises en compte, il faut ajouter des options (`autoreload`).

In [ ]:
# répertoire outils / fichier frontiere.py / plusieurs fonctions dans le fichier
from outils.frontiere import plot_frontiere

# prise en compte automatique des modifications du module (pratique quand on développe ses propres outils)
%load_ext autoreload
%autoreload 2

In [ ]:
# cas d'usage: la même donnée, trois modèles, trois frontières

plt.figure(figsize=(12,4), facecolor='white')
plt.subplot(1,3,1)                    # indicage foireux hérité de matlab :)
plot_frontiere(Xapp, Yapp, mod1)
plt.scatter(Xapp[:,0], Xapp[:,1], c=Yapp)
plt.title('Naive Bayes')
plt.subplot(1,3,2)
plot_frontiere(Xapp, Yapp, mod_SVM1)
plt.scatter(Xapp[:,0], Xapp[:,1], c=Yapp)
plt.title('SVC linéaire')
plt.subplot(1,3,3)
plot_frontiere(Xapp, Yapp, mod_SVM2)
plt.scatter(Xapp[:,0], Xapp[:,1], c=Yapp)
plt.title('SVC gaussien (gamma=10)')

#### Visualisation 3D de la fonction de décision

Pour les classifieurs qui possèdent la méthode `predict_proba` (tous ou presque... à condition d'avoir mis les
bonnes options à la création), j'ai développé une fonction `plot_mesh` qui permet de voir la fonction de
décision en 3D.

1. Importer cette fonction depuis mon module (vérifier éventuellement son existence dans `outils/frontiere.py`)
2. Ouvrir une figure 3D (code donné ci-dessous)
3. Utiliser la fonction. ATTENTION: elle suppose un problème à 2 classes et un modèle sachant faire
   `predict_proba` (d'où le `probability=True` du SVM ci-dessus).

In [ ]:
# 1. import de la fonction
from outils.frontiere import plot_mesh

# 2. ouverture d'une figure 3D (obligatoire: plot_mesh dessine dans les axes courants)
fig = plt.figure(facecolor='white', figsize=(10,8))
ax  = fig.add_subplot(projection='3d')

# 3. appel de la fonction
plot_mesh(Xapp, Yapp, mod_SVM2)
ax.set_title("P(classe 0 | x) pour le SVM gaussien")

<a id="sec-b4"></a>
### B.4 Introspection des modèles

Que valent les paramètres appris? Idéalement, que signifient-ils?

<span style="color:red">ATTENTION: les modèles ont évidemment des paramètres spécifiques => trouver les
paramètres à explorer = lire la documentation de chaque modèle</span>

1. On passe par la documentation pour savoir quoi regarder:<BR>
   e.g. https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html
2. On va chercher les paramètres pour les afficher

**Convention `sklearn`**: les attributs *appris* (donc disponibles seulement après le `fit`) se terminent par
un souligné `_`.

In [ ]:
# Naive Bayes: 2 caractéristiques => 2 gaussiennes 1D par classe (hypothèse d'indépendance)

print("variances (n_classes, n_features):\n", mod1.var_)  # .sigma_ sur les versions de sklearn < 1.2
print("moyennes  (n_classes, n_features):\n", mod1.theta_)
print("a priori des classes             :", mod1.class_prior_)

[Exercice] Dans les SVM, la régularisation fait que la solution ne repose que sur quelques points
(les *vecteurs supports*). Faire en sorte d'afficher les points qui supportent la solution dans le cas linéaire:

```python
mod = svm.SVC(kernel='linear')
```

<img src="fig/svm_support_1.png">

In [ ]:
# entourer les vecteurs supports d'un SVM linéaire
mod_lin = svm.SVC(kernel='linear')
mod_lin.fit(Xapp, Yapp)

plt.figure(facecolor='white')
plot_frontiere(Xapp, Yapp, mod_lin)
plt.scatter(Xapp[:,0], Xapp[:,1], c=Yapp)
#  TODO 
plt.title("SVM linéaire: frontière et vecteurs supports")

In [ ]:
# [Hors sujet pour le TP] Génération des figures du cours
# NB: on travaille sur des COPIES locales (Xs, Ys) pour ne pas écraser Xapp / Yapp
#  TODO 

<a id="sec-b5"></a>
### B.5 Spécificité des modèles

Afin de faciliter leur utilisation, tous les modèles disposent de `fit` et `predict` (et aussi, la plupart du
temps, de `predict_proba`)... Mais les modèles ont aussi des spécificités qui apparaissent dans la documentation.
> <span style="color:magenta">Vous devez exploiter la généricité des architectures ET la spécificité des modèles :)</span>

Par exemple, sur le modèle NB gaussien, il est possible d'obtenir la **vraisemblance jointe** de chaque point
évalué pour chaque classe (`predict_joint_log_proba`), ce qui ouvre des perspectives applicatives telles que le
rejet des points ambigus... ou la détection de points aberrants (vraisemblance faible pour *toutes* les classes).

Vous devriez obtenir quelque chose de la forme:
<img src="./fig/vraisemblance.png">

Note: évidemment, la vraisemblance n'est tracée que pour l'une des deux classes.

In [ ]:
# Afficher en couleur la vraisemblance des points d'apprentissage pour la classe 0
#  TODO 

<a id="sec-b6"></a>
### B.6 Jouons avec un arbre de décision

Les arbres de décision ont un statut particulier: à cheval entre l'apprentissage symbolique et statistique...
Un petit exercice autour de ce modèle.

La plupart des réponses se trouvent [ici](https://scikit-learn.org/stable/modules/tree.html)

Voici les questions:
1. Construire le modèle par défaut et l'entrainer
2. Visualiser la frontière de décision puis l'algorithme de décision... Et réfléchir à la manière dont
   fonctionne cette décision. Pourquoi considère-t-on que ce modèle est plus explicable que d'autres?
3. Entrainer et visualiser un arbre de profondeur limitée à 1 puis 2 et analyser le résultat

Note: vous devez obtenir quelque chose de la forme:

<img src="fig/tree_decision_c.png"> <img src="fig/tree_decision_2d.png"><BR>
Bonus: trouver le nom de l'option qui colorie les branches de l'arbre en fonction de la classe d'affectation.

In [ ]:
from sklearn import tree

# 1. construction & apprentissage du modèle (profondeur non contrainte)
#  TODO 

# 2a. visualisation de la frontière
plt.figure(facecolor='white')
plot_frontiere(Xapp, Yapp, mod_arbre)   # si votre modèle ne s'appelle pas mod_arbre => mettre à jour
plt.scatter(Xapp[:,0], Xapp[:,1], c=Yapp)
plt.title("Arbre de décision: frontière (des marches d'escalier!)")
# plt.savefig('fig/tree_decision_2d.png')

# 2b. dessin de l'algorithme de décision
#  TODO 

In [ ]:
# 3. entrainer et visualiser un arbre de profondeur 1 puis 2
#  TODO 

<a id="sec-b7"></a>
### B.7 Jouons avec le SVM

Comparer un classifieur SVM avec un noyau linéaire et un noyau gaussien.

1. Tracer les frontières de décision
2. Jouer avec la largeur de bande des gaussiennes (`gamma`)
3. Mettre en évidence le phénomène de sur-apprentissage (très rapide avec les gaussiennes trop serrées)

Afin de tirer parti des capacités non linéaires des SVM, nous allons travailler sur des données non séparables
linéairement: un damier de petits nuages, où deux nuages voisins n'ont pas la même étiquette.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split   # la version sklearn du découpage fait à la main en A.3

# damier 5x5: 25 nuages, étiquetés alternativement -1 / +1
centers      = [[float(i), float(j)] for i in range(5) for j in range(5)]
clusters_std = 0.2
Xnl, ynl = make_blobs(n_samples=100, centers=centers, cluster_std=clusters_std,
                      n_features=2, random_state=0)
ynl = (ynl % 2)*2 - 1     # numéro de nuage -> étiquette dans {-1, +1}

Xnl_app, Xnl_test, ynl_app, ynl_test = train_test_split(Xnl, ynl, test_size=0.33, random_state=0)
print("données non linéaires:", Xnl.shape, " app/test:", len(ynl_app), len(ynl_test))

plt.figure(facecolor='white')
plt.scatter(Xnl[:,0], Xnl[:,1], c=ynl)
plt.title("Données non séparables linéairement (damier)")

In [ ]:
# Comparer le noyau linéaire et le noyau gaussien pour différentes valeurs de gamma.
# Pour chaque modèle: tracer la frontière et afficher les performances en apprentissage ET en test.
#  TODO 

<a id="sec-c"></a>
## C. Modèles de l'état de l'art

Si les approches bayésiennes sont rapides et que les SVM forment une référence solide, il faut admettre
aujourd'hui l'efficacité redoutable des approches ensemblistes sur un grand nombre de taches (en particulier sur
les données tabulaires).

Parmi toutes les approches de bagging / boosting, deux familles s'illustrent particulièrement: les forêts
aléatoires et le gradient boosting.

1. Forêt aléatoire ([wikipedia](https://en.wikipedia.org/wiki/Random_forest))
    - les arbres comportent une part aléatoire dans leur construction (ils travaillent sur un sous-ensemble de
      variables et de données)... Ils sont donc individuellement assez faibles
    - le mécanisme de vote rend l'ensemble de la forêt très efficace
    - [le classifieur sklearn](https://scikit-learn.org/stable/modules/ensemble.html#forests-of-randomized-trees)
2. Gradient boosting
    - très grossièrement: il s'agit d'une forêt où les arbres sont ajoutés itérativement pour réduire les erreurs
      de la forêt à l'itération précédente
    - [l'implémentation sklearn](https://scikit-learn.org/stable/modules/ensemble.html#gradient-tree-boosting)
      est intéressante... Mais la plus efficace (rapide + accélération matérielle + bons hyper-paramètres par
      défaut + interprétation) est XGBoost [lien](https://xgboost.readthedocs.io/en/stable/)
    - il existe une interface sklearn pour XGBoost... Ça s'utilise donc comme tous les autres modèles
    - depuis 2022, il semble que `catboost` supplante XGBoost, notamment dans les cas où il y a des variables
      catégorielles

<img src="fig/xgboost_the_things.jpg">

<a id="sec-c1"></a>
### C.1 Premières expérimentations

Pour les deux modèles:
1. Entrainer le modèle de base
2. Visualiser la frontière de décision
3. Essayer de jouer avec les paramètres: au moins le nombre d'arbres et leur profondeur
4. [OPT] Reprendre les données USPS et comparer les performances de ces approches à celles du bayésien naïf
   (cf. exercice [E.6](#sec-e6))

In [ ]:
# Re-génération de données simples (2 nuages gaussiens) avec les outils sklearn
centers      = [[-2.0, -2.0], [2.0, 2.0]]
clusters_std = [1.5, 1.5]
X, y = make_blobs(n_samples=100, centers=centers, cluster_std=clusters_std,
                  n_features=2, random_state=0)                          # 100 pts, 2 classes, 2 dim
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=0)

plt.figure(facecolor='white')
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, marker='o', label="apprentissage")
plt.scatter(X_test[:,0],  X_test[:,1],  c=y_test,  marker='*', edgecolors='r', label="test")
plt.legend()
plt.title("Données de la partie C")

In [ ]:
# Forêt aléatoire sklearn
from sklearn.ensemble import RandomForestClassifier

mod = RandomForestClassifier(random_state=0)
mod.fit(X_train, y_train)

# évaluation quantitative
yhat = mod.predict(X_test)
print("perf modèle RF", np.where(yhat == y_test,1,0).mean())

# tracé de la frontière
plt.figure(facecolor='white')
plot_frontiere(X_train, y_train, mod)   # si votre modèle ne s'appelle pas mod => mettre à jour
plt.scatter(X_train[:,0], X_train[:,1], c=y_train)
plt.title("Forêt aléatoire (paramètres par défaut)")

In [ ]:
# Informations de base sur le modèle appris
print("nb arbres                : ", mod.n_estimators)
print("profondeur max demandée   : ", mod.max_depth)   # None = pas de contrainte... et donc?
# la vraie profondeur se lit sur les arbres appris (attribut estimators_)
print("profondeur réelle (10 1ers): ", [mod.estimators_[i].get_depth() for i in range(10)])

In [ ]:
# Jeu avec les paramètres de la forêt aléatoire (code donné)

mod1 = RandomForestClassifier(max_depth=1,  n_estimators=5,   random_state=0)  # très simple
mod1.fit(X_train, y_train)

mod2 = RandomForestClassifier(max_depth=10, n_estimators=100, random_state=0)  # beaucoup plus riche
mod2.fit(X_train, y_train)

# tracé des frontières
plt.figure(figsize=(9,4), facecolor='white')
plt.subplot(1,2,1)
plot_frontiere(X_train, y_train, mod1)
plt.scatter(X_train[:,0], X_train[:,1], c=y_train)
plt.title("5 arbres, profondeur 1 (test={:.2f})".format(mod1.score(X_test,y_test)))
plt.subplot(1,2,2)
plot_frontiere(X_train, y_train, mod2, step=40)
plt.scatter(X_train[:,0], X_train[:,1], c=y_train)
plt.title("100 arbres, profondeur 10 (test={:.2f})".format(mod2.score(X_test,y_test)))

# N'hésitez pas à complexifier les données (cf Xnl, ynl) et à relancer ce bout de code

In [ ]:
# XGBoost
import xgboost as xgb    # !pip install xgboost   # en cas de besoin

bst = xgb.XGBClassifier().fit(X_train, y_train)   # interface sklearn => fit / predict comme d'habitude
yhat = bst.predict(X_test)

print("perf modèle XGB", np.where(yhat == y_test,1,0).mean())

# tracé de la frontière
plt.figure(facecolor='white')
plot_frontiere(X_train, y_train, bst)   # si votre modèle ne s'appelle pas bst => mettre à jour
plt.scatter(X_train[:,0], X_train[:,1], c=y_train)
plt.title("XGBoost (paramètres par défaut)")

<a id="sec-c2"></a>
### C.2 Introspection: pondération des caractéristiques

Si on crée le jeu de données jouet suivant:

$$X = \begin{pmatrix}  x_{11}& x_{12} & x_{13} \sim \mathcal N(0,\sigma) & \ldots & x_{1d} \sim \mathcal N(0,\sigma) \\
x_{21}& x_{22} & x_{23} \sim \mathcal N(0,\sigma) & \ldots & x_{2d} \sim \mathcal N(0,\sigma) \\
\vdots& \vdots & \vdots & \ddots &\vdots \\
x_{n1}& x_{n2} & x_{n3} \sim \mathcal N(0,\sigma) & \ldots & x_{nd} \sim \mathcal N(0,\sigma) \\
\end{pmatrix} ,\qquad
Y = \begin{pmatrix}  y_{1} \\
y_{2}\\
\vdots\\
y_{n} \\
\end{pmatrix} ,\qquad y_i\in\{0,1\}
$$

Il s'agit d'un problème où les deux premières colonnes ont du sens et dans lesquelles on a ajouté des colonnes
de bruit blanc. La visualisation des deux premières colonnes avec les étiquettes donne classiquement:

<img src="fig/data2d.png">

La question qui se pose:
> <span style="color:magenta">Est-on capable de donner un score aux différentes colonnes, une fois
> l'apprentissage effectué?</span>

1. Avec les modèles linéaires, la réponse est triviale: il suffit de regarder les coefficients du classifieur
2. Avec une approche ensembliste, c'est possible aussi et c'est déjà implémenté: il suffit d'aller chercher les
   bonnes méthodes

Pour l'explication des combinaisons qui sont calculées (et les risques de sur-interprétation):
[lien](https://towardsdatascience.com/be-careful-when-interpreting-your-features-importance-in-xgboost-6e16132588e7)

In [ ]:
# données: 2 colonnes informatives + 20 colonnes de bruit
np.random.seed(42)     # reproductibilité du bruit

centers      = [[-2.0, -2.0], [2.0, 2.0]]
clusters_std = [1.5, 1.5]
X, y = make_blobs(n_samples=50, centers=centers, cluster_std=clusters_std,
                  n_features=2, random_state=0)          # 50 pts, 2 classes, 2 dim informatives

ndim_noise = 20
Noise = np.random.randn(len(X), ndim_noise)*clusters_std[0]
Xn = np.concatenate((X, Noise), axis=1)                  # (50, 22)
print("dimensions du problème bruité:", Xn.shape)

X_train, X_test, y_train, y_test = train_test_split(Xn, y, test_size=0.33, random_state=0)

In [ ]:
# les 2 colonnes informatives (= le problème que l'on saurait résoudre à l'oeil)
plt.figure(facecolor='white')
plt.scatter(X[:,0], X[:,1], c=y)
plt.xlabel("$X_1$")
plt.ylabel("$X_2$")
plt.title("Les 2 caractéristiques informatives sur 22")
# plt.savefig("fig/data2d.png")

In [ ]:
from sklearn.linear_model import LogisticRegression

# apprentissage d'un modèle linéaire et d'un modèle ensembliste sur les MÊMES données bruitées
mod1 = LogisticRegression()
mod2 = xgb.XGBClassifier()

mod1.fit(X_train, y_train)
mod2.fit(X_train, y_train)

print("perf test: régression logistique = {:.2f} / XGBoost = {:.2f}".format(
      mod1.score(X_test, y_test), mod2.score(X_test, y_test)))

Pour le modèle linéaire, vous devez être capable:
1. d'aller chercher les paramètres appris
2. de tracer un diagramme `plt.bar` de ces paramètres
    - à vous de choisir si vous optez pour la valeur absolue ou pas: les deux choix sont argumentables
    - vous ferez attention à la dimension des paramètres

<img src="fig/bar_mod_lin.png">

In [ ]:
# affichage des poids associés aux caractéristiques du problème
#  TODO 

Faire la même chose avec XGBoost en allant chercher la (les) fonction(s) utile(s) dans la documentation, puis
comparer les deux diagrammes: quel modèle fait le mieux le tri entre signal et bruit?

In [ ]:
# affichage de l'importance des caractéristiques pour XGBoost
#  TODO 

<a id="sec-d"></a>
## D. Extension de scikit-learn

Selon les principes de la programmation objet, vous pouvez définir votre propre classifieur et utiliser ensuite
toutes les fonctions qui attendent un classifieur (validation croisée, grid search, pipeline...).

Voici un code minimaliste illustrant l'héritage en python (très simple) dans le cas sklearn.
Les deux règles à respecter:
1. `__init__` ne fait que **stocker les hyper-paramètres** (pour que `clone` et la sérialisation fonctionnent)
2. tout ce qui est **appris** est rangé dans `fit`, dans des attributs terminés par `_`

Pour plus de détails: [lien](https://scikit-learn.org/stable/developers/develop.html)

In [ ]:
# un classifieur à poids aléatoires fixes (un peu absurde donc...) mais qui montre comment ça marche
from sklearn.base import BaseEstimator, ClassifierMixin

class LinearFixClassifier(BaseEstimator, ClassifierMixin):

    def __init__(self, data_dim=2, random_state=0):
        # RÈGLE 1: on ne fait que stocker les hyper-paramètres, aucun calcul ici
        self.data_dim     = data_dim
        self.random_state = random_state

    def fit(self, X, y):
        # notre modèle n'apprend rien... on se contente de tirer des poids au hasard
        rng      = np.random.RandomState(self.random_state)
        self.w_  = rng.randn(self.data_dim)   # RÈGLE 2: attribut appris => souligné final
        # mémorisation des classes vues pendant l'apprentissage (convention sklearn)
        self.classes_ = np.unique(y)
        return self                           # fit retourne toujours self (pour chainer les appels)

    def predict(self, X):   # dans le cas binaire seulement
        # mon classifieur sort un signe: je le convertis vers les étiquettes du problème
        return np.where(X @ self.w_ > 0, self.classes_[1], self.classes_[0])

À vous de vérifier que vous pouvez invoquer une validation croisée sur ce classifieur !

C'est tout l'intérêt de respecter l'interface: une fonction que vous n'avez pas écrite (`cross_val_score`) est
capable de manipuler un objet qu'elle ne connait pas.

In [ ]:
# données
X, y = make_blobs(n_samples=50, centers=[[-2.0,-2.0], [2.0,2.0]], cluster_std=[1.5,1.5],
                  n_features=2, random_state=0)

mod = LinearFixClassifier(data_dim=2)
mod.fit(X, y)
yhat = mod.predict(X)

print("prédictions :", yhat)
print("perf        : {:.2f}".format(mod.score(X, y)))   # score vient de ClassifierMixin: gratuit!

# à ajouter: la validation croisée (plus ambitieux => une autre fonction, que je ne connais pas,
# va utiliser mon objet!)

In [ ]:
from sklearn.model_selection import cross_val_score

n_fold = 5
scores = cross_val_score(mod, X, y, cv=n_fold, scoring='accuracy')  # tout est caché dedans :)
print("scores des {} folds : {}".format(n_fold, scores))
print("moyenne +/- écart-type : {:.2f} +/- {:.2f}".format(scores.mean(), scores.std()))

# NB: cross_val_score CLONE le modèle avant chaque fold (il repart donc des hyper-paramètres de __init__
#     et rappelle fit): c'est exactement pour cela que les poids ne doivent PAS être tirés dans __init__.

<a id="sec-e"></a>
## E. Exercices complémentaires

Les exercices de cette partie sont indépendants les uns des autres (chacun régénère ses propres données) et
peuvent être traités dans le désordre, selon le temps disponible. Ils reprennent tous la même mécanique
`création / fit / predict / score`, mais chacun met en lumière un réflexe méthodologique différent.

| Exercice | Notion travaillée |
|---|---|
| E.1 | un nouveau modèle: les k plus proches voisins, et le compromis biais / variance |
| E.2 | l'influence de la mise à l'échelle des données, et les `Pipeline` |
| E.3 | la courbe de complexité: diagnostiquer sous- et sur-apprentissage |
| E.4 | validation croisée et recherche automatique d'hyper-paramètres |
| E.5 | la même API pour un autre problème: la régression |
| E.6 | synthèse: comparer des modèles sur des données réelles (USPS) |

<a id="sec-e1"></a>
### E.1 Les k plus proches voisins

Le modèle des k plus proches voisins (`sklearn.neighbors.KNeighborsClassifier`) n'apprend rien: il se contente
de mémoriser les données et de faire voter les `k` points les plus proches au moment de la prédiction.

1. Entrainer ce modèle sur les données jouets de la partie A pour `k` = 1, 5, 15 et 50
2. Tracer les frontières de décision et afficher les performances en apprentissage et en test
3. Répondre: pourquoi le taux de bonne classification en apprentissage vaut-il exactement 1 pour `k=1`?
   Que se passe-t-il quand `k` s'approche du nombre de points d'apprentissage?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# on réutilise les données jouets Xapp/Yapp, Xtest/Ytest de la partie A
#  TODO 

<a id="sec-e2"></a>
### E.2 Mise à l'échelle des données et pipelines

Beaucoup de modèles reposent sur une **distance** entre individus (kNN, SVM à noyau gaussien, ...): ils sont
donc très sensibles aux unités de mesure. D'autres (les arbres) travaillent variable par variable et y sont
totalement insensibles.

1. Repartir des données à 2 nuages gaussiens et multiplier artificiellement la 2e caractéristique par 500
   (comme si on passait de mètres à millimètres sur cette seule variable)
2. Évaluer en validation croisée un kNN, un SVM et un arbre de décision sur ces données déséquilibrées
3. Recommencer en insérant un `StandardScaler` **dans un `Pipeline`** (`sklearn.pipeline.make_pipeline`)
4. Répondre: quels modèles sont affectés? Pourquoi faut-il impérativement mettre la normalisation dans le
   pipeline plutôt que de normaliser les données une fois pour toutes avant la validation croisée?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

# données: 2 nuages, dont la seconde caractéristique est "en millimètres"
Xe, ye = make_blobs(n_samples=200, centers=[[-2.0,-2.0], [2.0,2.0]], cluster_std=[1.5,1.5],
                    n_features=2, random_state=0)
Xe_desequilibre = Xe.copy()
Xe_desequilibre[:,1] *= 500

#  TODO 

<a id="sec-e3"></a>
### E.3 Courbe de complexité: sous-apprentissage et sur-apprentissage

On a vu le sur-apprentissage apparaitre sur le SVM (B.7). Formalisons le diagnostic sur un arbre de décision,
dont la complexité se règle très simplement avec `max_depth`.

1. Générer des données `make_moons(n_samples=400, noise=0.35, random_state=0)` et les séparer en app/test
2. Pour `max_depth` de 1 à 15, mémoriser le taux de bonne classification en apprentissage et en test
3. Tracer les deux courbes sur la même figure et repérer la profondeur optimale
4. Répondre: dans quelle zone est-on en sous-apprentissage? en sur-apprentissage? Que vaut la performance en
   apprentissage quand la profondeur augmente, et pourquoi?

In [ ]:
from sklearn.datasets import make_moons

Xm, ym = make_moons(n_samples=400, noise=0.35, random_state=0)
Xm_app, Xm_test, ym_app, ym_test = train_test_split(Xm, ym, test_size=0.5, random_state=0)

plt.figure(facecolor='white')
plt.scatter(Xm[:,0], Xm[:,1], c=ym)
plt.title("make_moons: 2 croissants bruités")

#  TODO 

<a id="sec-e4"></a>
### E.4 Validation croisée et recherche d'hyper-paramètres

Régler `gamma` à la main (B.7) en regardant la performance de **test** est méthodologiquement faux: on finit par
choisir un modèle qui colle aux données de test, et l'estimation de performance devient optimiste.
La bonne pratique consiste à régler les hyper-paramètres en validation croisée **sur les données
d'apprentissage uniquement**, puis à n'utiliser le test qu'une seule fois, à la fin.

1. Reprendre les données non linéaires `Xnl_app, ynl_app` de la partie B.7
2. Utiliser `GridSearchCV` pour explorer conjointement `C` (dans `[0.1, 1, 10, 100]`) et
   `gamma` (dans `[0.01, 0.1, 1, 10, 100]`) d'un `svm.SVC`
3. Afficher les meilleurs hyper-paramètres, le score de validation croisée associé, puis le score final en test
4. Bonus: afficher la carte des scores de validation croisée (`cv_results_['mean_test_score']`) sous forme
   d'image `plt.imshow` pour visualiser le paysage des hyper-paramètres

In [ ]:
from sklearn.model_selection import GridSearchCV

#  TODO 

<a id="sec-e5"></a>
### E.5 La même API pour la régression

Tout ce que nous avons vu se transpose à la **régression** (prédire une valeur continue et non une classe):
mêmes `fit` / `predict`, mêmes modèles ensemblistes, seule la métrique change (`score` renvoie ici le
coefficient de détermination $R^2$, et non un taux de bonne classification).

1. Charger le jeu `datasets.load_diabetes()` et le séparer en apprentissage / test
2. Comparer une `LinearRegression` et un `RandomForestRegressor` ($R^2$ en apprentissage et en test)
3. Tracer le nuage (valeur prédite, valeur réelle) sur les données de test pour le meilleur modèle
4. Répondre: que vaut un $R^2$ de 0? un $R^2$ négatif est-il possible?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

#  TODO 

<a id="sec-e6"></a>
### E.6 Synthèse: comparaison de modèles sur USPS

Reprenons les données USPS (256 dimensions, 10 classes) chargées en B.1 pour un dernier tour d'horizon.

1. Comparer sur les mêmes données: `GaussianNB`, `SVC`, `RandomForestClassifier` et `XGBClassifier`
   (taux de bonne classification en test **et** temps d'apprentissage: les deux comptent en pratique)
2. Afficher la matrice de confusion (`ConfusionMatrixDisplay`) du meilleur modèle
3. Afficher quelques images mal classées (`Xu_test[i].reshape(16,16)`) et juger de leur difficulté
4. Répondre: quelles paires de chiffres sont les plus confondues? La performance du bayésien naïf
   s'explique-t-elle par son hypothèse d'indépendance des pixels?

In [ ]:
import time
from sklearn.metrics import ConfusionMatrixDisplay

#  TODO 

----
<a id="sec-annexe"></a>
## Annexe: transformation du notebook en version étudiante

La cellule ci-dessous produit `1-notebook-intro.ipynb` à partir de ce fichier: tout ce qui se trouve entre les
balises de correction est remplacé par un `TODO`.

**À faire avant de la lancer**: `Kernel > Restart & Clear Output`, sinon les sorties des cellules de correction
(résultats et figures) restent visibles dans la version distribuée.

In [ ]:
#  TODO 